# MSWEP on Google Drive

Scratch notebook for looking at the shared MSWEP folder before and after a download,
the same role `gleam_search.ipynb` plays for GLEAM. Run it from the repo root so the
`utils` imports resolve.

Prerequisite: an rclone remote pointing at the shared Drive (see the README).

In [ ]:
import os
import sys
import posixpath
import subprocess
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

sys.path.insert(0, os.getcwd())
from utils.path_utils import load_config
from utils.rclone_utils import build_flags, remote_path, lsjson

settings = load_config('config/config_download.yaml')
rclone_settings = settings['rclone']
flags = build_flags(rclone_settings)
print('remote:', remote_path(rclone_settings))
print('flags: ', ' '.join(flags))

## 1. What has actually been shared

`--drive-shared-with-me` is a separate namespace from My Drive. If the folder does not
show up here, either the share has not been accepted or `shared_with_me` is wrong in
the config.

In [ ]:
def rclone(*args):
    """Run an rclone command and return its stdout."""
    result = subprocess.run(['rclone', *args], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'rclone failed (exit {result.returncode}): {result.stderr.strip()}')
    return result.stdout


# top level of the share; use this to confirm the folder name in the config
print(rclone('lsd', f'{rclone_settings["remote"]}:', *flags))

In [ ]:
# one level inside the configured root
print(rclone('lsd', remote_path(rclone_settings), *flags))

## 2. What each product directory holds

One recursive listing of the whole root, then counts and volumes grouped by directory.
This is the number to sanity check the download against.

In [ ]:
entries = lsjson(remote_path(rclone_settings), flags)
print(f'{len(entries)} files under {remote_path(rclone_settings)}')

by_directory = defaultdict(lambda: {'files': 0, 'bytes': 0})
for entry in entries:
    directory = posixpath.dirname(entry['Path']) or '.'
    by_directory[directory]['files'] += 1
    by_directory[directory]['bytes'] += entry['Size']

summary = pd.DataFrame([
    {'directory': directory, 'files': counts['files'], 'GB': counts['bytes'] / 1024**3}
    for directory, counts in sorted(by_directory.items())
])
summary['mean_MB'] = summary['GB'] * 1024 / summary['files']
summary.sort_values('GB', ascending=False)

## 3. `Past/Daily` in detail

Filenames are `YYYYDOY.nc`. Parse them into a date index so the record's extent, any
gaps, and the file size distribution are all visible. Gaps found here are expected to
be gaps after the download too, so note them before blaming the transfer.

In [ ]:
product = 'Past/Daily'
daily = lsjson(remote_path(rclone_settings, product), flags)
print(f'{len(daily)} entries in {product}')

records = []
unparsed = []
for entry in daily:
    name = posixpath.basename(entry['Path'])
    stem, extension = posixpath.splitext(name)
    if extension != '.nc' or len(stem) != 7 or not stem.isdigit():
        unparsed.append(name)
        continue
    year, doy = int(stem[:4]), int(stem[4:])
    records.append({
        'name': name,
        'date': pd.Timestamp(year=year, month=1, day=1) + pd.Timedelta(days=doy - 1),
        'year': year,
        'MB': entry['Size'] / 1024**2,
    })

files = pd.DataFrame(records).sort_values('date').reset_index(drop=True)
print(f'parsed {len(files)}, skipped {len(unparsed)}: {unparsed[:10]}')
print(f'first {files["date"].min().date()}  last {files["date"].max().date()}')
print(f'total {files["MB"].sum() / 1024:.1f} GB, mean {files["MB"].mean():.1f} MB per file')

In [ ]:
# missing days between the first and last date present
expected = pd.date_range(files['date'].min(), files['date'].max(), freq='D')
gaps = expected.difference(pd.DatetimeIndex(files['date']))
print(f'{len(gaps)} missing days out of {len(expected)}')
if len(gaps):
    # collapse runs of consecutive missing days so long outages read as one line
    breaks = np.where(np.diff(gaps.values).astype('timedelta64[D]').astype(int) > 1)[0]
    for start, end in zip(np.r_[0, breaks + 1], np.r_[breaks, len(gaps) - 1]):
        first, last = gaps[start].date(), gaps[end].date()
        print(f'  {first} .. {last}  ({end - start + 1} days)' if first != last else f'  {first}')

In [ ]:
# files per year, and any year whose count does not match its calendar length
per_year = files.groupby('year').agg(files=('name', 'size'), GB=('MB', lambda mb: mb.sum() / 1024))
per_year['expected'] = [366 if pd.Timestamp(y, 1, 1).is_leap_year else 365 for y in per_year.index]
per_year['short_by'] = per_year['expected'] - per_year['files']
per_year

In [ ]:
# size distribution; a cluster of very small files usually means truncated uploads
print(files['MB'].describe())
files.nsmallest(10, 'MB')[['name', 'date', 'MB']]

## 4. After the download

Compare the local tree against the remote. `verify_downloads` in `mswep_download.py`
already does a size check at the end of every run; this is the deeper check, and it
reads every byte, so scope it to a subset unless you have time to spare.

In [ ]:
from mswep_download import download_root, format_product

local = os.path.join(download_root(settings), format_product(product))
on_disk = sorted(f for f in os.listdir(local) if f.endswith('.nc')) if os.path.isdir(local) else []
print(f'{len(on_disk)} local files vs {len(files)} remote')
missing = set(files['name']) - set(on_disk)
print(f'{len(missing)} missing: {sorted(missing)[:10]}')

In [ ]:
# full checksum comparison, downloading each file again to hash it -- slow, so only
# worth running over one year at a time
# print(rclone('check', remote_path(rclone_settings, product), local, '--download', '--one-way', *flags))

In [ ]:
import xarray as xr

# open one file and look at what is actually in it
sample = xr.open_dataset(os.path.join(local, on_disk[0]))
sample